# Оптимизация гиперпараметров

In [1]:
import sys

from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import config

In [2]:
from init_pipeline import init_experiment
init_experiment(project_root, config)

Инициализация пропущена: /Users/romansafronenkov/Documents/Projects/uplift_modeling_pipeline/artifacts/boosting_pipeline/status.json уже существует и init=True


## Логирование

In [3]:
import os

from src.utils.logger import setup_logging

LOG_DIR = project_root / 'artifacts' / config.general.experiment_name / 'log'
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = LOG_DIR / 'optimization.txt'

_logger = setup_logging(LOG_FILE, 'optimization')

/Users/romansafronenkov/Documents/Projects/uplift_modeling_pipeline/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Импорты

In [4]:
# ждем пока выполнится предыдущий ноутбук
import json
import time

while True:
    with open(project_root / 'artifacts' / config.general.experiment_name / 'status.json', 'r') as f:
        status = json.load(f)
    if status['choose_model']:
        _logger.info('Модель выбрана, оптимизируем гиперпараметры')
        break
    time.sleep(60) 

[2026-09-07 11:12:28,195] - [optimization] - [INFO] - Модель выбрана, оптимизируем гиперпараметры


In [5]:
import shutil
import copy
import json
import random
import time
from functools import partial

import joblib

from IPython.display import clear_output, display

import pandas as pd
import numpy as np

import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold, TimeSeriesSplit

from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.metrics import roc_auc_score, classification_report

from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder

from sklift.models import SoloModel, TwoModels
from causalml.inference.meta import BaseXClassifier
from causalml.propensity import compute_propensity_score

from sklift.metrics import qini_auc_score, uplift_at_k

import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [6]:
from src.preprocessing import preprocess_sdf
from src.utils.uplift_clf_metrics import show_results
from src.utils.optimization import metric_stability

In [7]:
if config.general.load_venv_to_spark:
    os.environ['PYSPARK_PYTHON'] = './environment/bin/python'
    os.environ['PYSPARK_DRIVER_PYTHON'] = './environment/bin/python'

In [8]:
np.random.seed(config.general.seed)
random.seed(config.general.seed)

In [9]:
from src.utils.get_spark import get_conf, get_spark

conf = get_conf()

if config.general.load_venv_to_spark:
    conf.set('spark.archives', project_root / 'venv.tar.gz#environment')

In [10]:
spark = get_spark(conf, app_name=config.general.spark_session_name)

clear_output()
spark

## Константы

In [17]:
_logger.info(f"Experiment name: {config.general.experiment_name}")
data_dir = project_root / 'data'
fs_path = project_root / 'artifacts' / config.general.experiment_name / 'feature_selection'
choose_model_path = project_root / 'artifacts' / config.general.experiment_name / 'choose_model'
optimize_path = project_root / 'artifacts' / config.general.experiment_name / 'optimize'

os.makedirs(optimize_path, exist_ok=True)

[2026-09-07 11:13:35,065] - [optimization] - [INFO] - Experiment name: boosting_pipeline


In [12]:
if config.general.optimize:
    with open(fs_path / 'features_selected_final.json', 'r') as f:
        features = json.load(f)
    _logger.info(f'Признаки загружены: {len(features)}')

    with open(choose_model_path / 'choosen_model.json', 'r') as f:
        choosen_model = json.load(f)

    _logger.info(f'Choosen model is {choosen_model}')

[2026-09-07 11:12:44,128] - [optimization] - [INFO] - Признаки загружены: 25
[2026-09-07 11:12:44,129] - [optimization] - [INFO] - Choosen model is slearner


## Загрузка и предобработка данных

In [14]:
if config.general.optimize:
    features_to_select = []
    for feature in features:
        if feature.endswith('_diff'):
            features_to_select.append(feature.replace('_diff', ''))
            continue
        if feature.endswith('_sum'):
            features_to_select.append(feature.replace('_sum', ''))
            continue
        features_to_select.append(feature)

In [18]:
if config.general.optimize:
    _logger.info('Загружаем датасет')
    dataset = spark.read.parquet(str(data_dir / 'train_dataset.parquet'))
    dataset = dataset.select(features_to_select + [config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col])

    len_dataset = dataset.count()
    _logger.info(f'Dataset length: {len_dataset}, dataset columns: {len(dataset.columns)}')

    dataset, dt_features, bitmask_cols = preprocess_sdf(dataset, config)
    dataset = dataset.select(features+[config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col])

    def find_string_features(sdf):
        def find_str(pair):
            col, dtype = pair
            if col not in [config.dataset.date_col, config.dataset.target_col, config.dataset.treatment_col]:
                if dtype == 'string':
                    return col
        return [pair[0] for pair in list(filter(find_str, sdf.dtypes))]

    string_features = find_string_features(dataset)
    string_cols_to_encode = [col for col in features if col in string_features]
    _logger.info(f"Строковых признаков: {len(string_cols_to_encode)}")

    train_dataset = dataset.toPandas()

[2026-09-07 11:13:37,238] - [optimization] - [INFO] - Загружаем датасет
[2026-09-07 11:13:39,073] - [optimization] - [INFO] - Dataset length: 458976, dataset columns: 28
[2026-09-07 11:13:39,073] - [src.preprocessing] - [INFO] - Num of datetime features: 0
[2026-09-07 11:13:39,074] - [src.preprocessing] - [INFO] - Num of bitmask features: 0
[2026-09-07 11:13:39,111] - [src.preprocessing] - [INFO] - Num of decimal features: 0
[2026-09-07 11:13:39,112] - [src.preprocessing] - [INFO] - Unique datatypes in dataset: {'int', 'double', 'string'}
[2026-09-07 11:13:39,139] - [optimization] - [INFO] - Строковых признаков: 0


26/09/07 11:13:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [19]:
spark.stop()

## Оптимизация гиперпараметров

In [32]:
if config.general.optimize:
    if config.optimization.metric_to_optimize.value == 'qini':
        metric_func = qini_auc_score
    elif config.optimization.metric_to_optimize.value == 'uplift_at_k':
        metric_func = partial(uplift_at_k, k=config.optimization.k, strategy='by_group')

    key_metric_name = config.optimization.metric_to_optimize.value
    if key_metric_name == 'uplift_at_k':
        key_metric_name = key_metric_name.replace('k', str(int(config.optimization.k*100)))

In [33]:
if config.general.optimize:
    X, y = train_dataset[features+[config.dataset.treatment_col]], train_dataset[config.dataset.target_col]
    dates = train_dataset[config.dataset.date_col]
    y_strat = X[config.dataset.treatment_col].astype(str) + '_' + dates.astype(str) + '_' + y.astype(str)

    kfold = StratifiedKFold(n_splits=4, shuffle=True, random_state=config.general.seed)

    if config.optimization.use_time_series_split:

        class TimeSeriesKFold:
            def __init__(self, dates, n_splits=4, test_size=1, max_train_size=4, gap=1):
                self.ts = TimeSeriesSplit(
                    n_splits=n_splits,
                    test_size=test_size,
                    max_train_size=max_train_size,
                    gap=gap
                )
                self.dates = dates
                self.unique_dates = dates.unique()
                self.unique_dates.sort()

            def split(self, X, y=None, groups=None):
                for train_d_idx, val_d_idx in self.ts.split(self.unique_dates):
                    train_dates = self.unique_dates[train_d_idx]
                    val_dates = self.unique_dates[val_d_idx]

                    yield (self.dates[self.dates.isin(train_dates)].index, self.dates[self.dates.isin(val_dates)].index)

        kfold = TimeSeriesKFold(dates)

    else:
        kfold = StratifiedKFold(n_splits=4, shuffle=True, random_state=config.general.seed)

In [34]:
if config.general.optimize and choosen_model == 'slearner':
    def save_params(study, path):
        params = study.best_params
        params['bootstrap_type'] = 'Bernoulli'
        params['random_state'] = config.general.seed
        params['verbose'] = False
        params['loss_function'] = f"Focal:focal_alpha={params['focal_alpha']};focal_gamma={params['focal_gamma']}"
        del params['focal_alpha'], params['focal_gamma']

        SAVE_PARAMS_NAME = path / f'{config.optimization.metric_to_optimize.value}_best_params.pkl'

        joblib.dump(params, SAVE_PARAMS_NAME)

        # save study
        SAVE_STUDY_NAME = path / f'{config.optimization.metric_to_optimize.value}_study.pkl'
        joblib.dump(study, SAVE_STUDY_NAME)


    # n_trials = 2 ** num_hyperparams + warmup ~20%
    def objective(trial):
        params = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves', 2, 256),
            'max_depth': trial.suggest_int('max_depth', 2, 10),
            'iterations': trial.suggest_int('iterations', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 250),
            # 'auto_class_weights': trial.suggest_categorical('auto_class_weights', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        focal_alpha = trial.suggest_float('focal_alpha', 0.1, 0.9, step=0.1)
        focal_gamma = trial.suggest_float('focal_gamma', 0.5, 3, step=0.5)

        params['loss_function'] = f"Focal:focal_alpha={focal_alpha};focal_gamma={focal_gamma}"

        metrics = []

        for (train_ix, val_ix) in kfold.split(X, y_strat):
            x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
            x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

            if len(string_cols_to_encode):
                encoder = TargetEncoder()
                x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
                x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

            model = SoloModel(estimator=CatBoostClassifier(**params))
            model.fit(x_train[features], y_train, x_train[config.dataset.treatment_col])

            uplift_train = model.predict(x_train[features])
            uplift_val = model.predict(x_val[features])

            if all(uplift_val == 0):
                raise optuna.TrialPruned()

            metric_train = metric_func(y_train.values, uplift_train.ravel(), x_train[config.dataset.treatment_col].values)
            metric_val = metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values)

            metric = metric_stability(metric_train, metric_val, 0.2)
            metrics.append(metric)

        _logger.info(f'Trial #{trial.number}. Metrics: {metrics} (mean={np.mean(metrics):.4f}). Params: {trial.params}')
        return np.mean(metrics)

In [35]:
if config.general.optimize and choosen_model == 'tlearner':
    def save_params(study, path):
        params = study.best_params

        params_tr = {
            'loss_function': f"Focal:focal_alpha={params['focal_alpha_tr']};focal_gamma={params['focal_gamma_tr']}",
            'bootstrap_type': 'Bernoulli',
            'random_state': config.general.seed,
            'verbose': False
        }
        params_c = {
            'loss_function': f"Focal:focal_alpha={params['focal_alpha_c']};focal_gamma={params['focal_gamma_c']}",
            'bootstrap_type': 'Bernoulli',
            'random_state': config.general.seed,
            'verbose': False
        }
        del params['focal_alpha_tr'], params['focal_gamma_tr'], params['focal_alpha_c'], params['focal_gamma_c']

        for key, value in params.items():
            if key.endswith('_c'):
                params_c[key[:-2]] = value
            elif key.endswith('_tr'):
                params_tr[key[:-3]] = value

        SAVE_PARAMS_NAME_TR = path / f'{config.optimization.metric_to_optimize.value}_best_params_tr.pkl'
        SAVE_PARAMS_NAME_C = path / f'{config.optimization.metric_to_optimize.value}_best_params_c.pkl'

        joblib.dump(params_tr, SAVE_PARAMS_NAME_TR)
        joblib.dump(params_c, SAVE_PARAMS_NAME_C)

        # save study
        SAVE_STUDY_NAME = path / f'{config.optimization.metric_to_optimize.value}_study.pkl'
        joblib.dump(study, SAVE_STUDY_NAME)


    def objective(trial):
        params_tr = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_tr', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_tr', 2, 256),
            'max_depth': trial.suggest_int('max_depth_tr', 2, 10),
            'iterations': trial.suggest_int('iterations_tr', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_tr', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_tr', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_tr', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_tr', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_tr', 1, 250),
            # 'auto_class_weights': trial.suggest_categorical('auto_class_weights_tr', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_c = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_c', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_c', 2, 256),
            'max_depth': trial.suggest_int('max_depth_c', 2, 10),
            'iterations': trial.suggest_int('iterations_c', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_c', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_c', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_c', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_c', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_c', 1, 250),
            # 'auto_class_weights': trial.suggest_categorical('auto_class_weights_c', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        focal_alpha_tr = trial.suggest_float('focal_alpha_tr', 0.1, 0.9, step=0.1)
        focal_gamma_tr = trial.suggest_float('focal_gamma_tr', 0.5, 3, step=0.5)

        focal_alpha_c = trial.suggest_float('focal_alpha_c', 0.1, 0.9, step=0.1)
        focal_gamma_c = trial.suggest_float('focal_gamma_c', 0.5, 3, step=0.5)

        params_tr['loss_function'] = f"Focal:focal_alpha={focal_alpha_tr};focal_gamma={focal_gamma_tr}"
        params_c['loss_function'] = f"Focal:focal_alpha={focal_alpha_c};focal_gamma={focal_gamma_c}"

        metrics = []

        for (train_ix, val_ix) in kfold.split(X, y_strat):
            x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
            x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

            if len(string_cols_to_encode):
                encoder = TargetEncoder()
                x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
                x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

            model = TwoModels(
                estimator_trmnt=CatBoostClassifier(**params_tr),
                estimator_ctrl=CatBoostClassifier(**params_c)
            )
            model.fit(x_train[features], y_train, x_train[config.dataset.treatment_col])

            uplift_train = model.predict(x_train[features])
            uplift_val = model.predict(x_val[features])

            if all(uplift_val == 0):
                raise optuna.TrialPruned()

            metric_train = metric_func(y_train.values, uplift_train.ravel(), x_train[config.dataset.treatment_col].values)
            metric_val = metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values)

            metric = metric_stability(metric_train, metric_val, 0.2)
            metrics.append(metric)
            
        _logger.info(f'Trial #{trial.number}. Metrics: {metrics} (mean={np.mean(metrics):.4f}). Params: {trial.params}')
        return np.mean(metrics)

In [36]:
if config.general.optimize and choosen_model == 'xlearner':
    def save_params(study, path):
        params = study.best_params

        params_co = {
            'loss_function': f"Focal:focal_alpha={params['focal_alpha_co']};focal_gamma={params['focal_gamma_co']}",
            'bootstrap_type': 'Bernoulli',
            'random_state': config.general.seed,
            'verbose': False
        }
        params_to = {
            'loss_function': f"Focal:focal_alpha={params['focal_alpha_to']};focal_gamma={params['focal_gamma_to']}",
            'bootstrap_type': 'Bernoulli',
            'random_state': config.general.seed,
            'verbose': False
        }
        params_ce = {
            'bootstrap_type': 'Bernoulli',
            'random_state': config.general.seed,
            'verbose': False
        }
        params_te = {
            'bootstrap_type': 'Bernoulli',
            'random_state': config.general.seed,
            'verbose': False
        }
        del params['focal_alpha_to'], params['focal_gamma_to'], params['focal_alpha_co'], params['focal_gamma_co']

        for key, value in params.items():
            if key.endswith('_co'):
                params_co[key[:-3]] = value
            elif key.endswith('_to'):
                params_to[key[:-3]] = value
            elif key.endswith('_ce'):
                params_ce[key[:-3]] = value
            elif key.endswith('_te'):
                params_te[key[:-3]] = value

        SAVE_PARAMS_NAME_CO = path / f'{config.optimization.metric_to_optimize.value}_best_params_co.pkl'
        SAVE_PARAMS_NAME_TO = path / f'{config.optimization.metric_to_optimize.value}_best_params_to.pkl'
        SAVE_PARAMS_NAME_CE = path / f'{config.optimization.metric_to_optimize.value}_best_params_ce.pkl'
        SAVE_PARAMS_NAME_TE = path / f'{config.optimization.metric_to_optimize.value}_best_params_te.pkl'

        joblib.dump(params_co, SAVE_PARAMS_NAME_CO)
        joblib.dump(params_to, SAVE_PARAMS_NAME_TO)
        joblib.dump(params_ce, SAVE_PARAMS_NAME_CE)
        joblib.dump(params_te, SAVE_PARAMS_NAME_TE)

        # save study
        SAVE_STUDY_NAME = path / f'{config.optimization.metric_to_optimize.value}_study.pkl'
        joblib.dump(study, SAVE_STUDY_NAME)\

    FIT_PROPENSITY = not 0.485 <= X[config.dataset.treatment_col].mean() <= 0.515

    class DummyPropensityModel:
        def predict(self, x):
            return np.full(shape=(x.shape[0],), fill_value=0.5)

    cv_idxs = []
    propensity_models = []

    for train_ix, val_ix in kfold.split(X, y_strat):
        cv_idxs.append((train_ix, val_ix))

        x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
        x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

        x_train = x_train.assign(**{config.dataset.target_col: y_train})
        x_val = x_val.assign(**{config.dataset.target_col: y_val})

        if len(string_cols_to_encode):
            encoder = TargetEncoder()
            x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y_train)
            x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

        if FIT_PROPENSITY:
            p, propensity_model = compute_propensity_score(
                x_train[features].values,
                x_train[config.dataset.treatment_col].values,
                calibrate_p=False
            )
        else:
            propensity_model = DummyPropensityModel()

        propensity_models.append(propensity_model)

    def objective(trial):
        params_co = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_co', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_co', 2, 256),
            'max_depth': trial.suggest_int('max_depth_co', 2, 10),
            'iterations': trial.suggest_int('iterations_co', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_co', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_co', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_co', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_co', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_co', 1, 250),
            # 'auto_class_weights': trial.suggest_categorical('auto_class_weights_co', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_to = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_to', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_to', 2, 256),
            'max_depth': trial.suggest_int('max_depth_to', 2, 10),
            'iterations': trial.suggest_int('iterations_to', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_to', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_to', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_to', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_to', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_to', 1, 250),
            # 'auto_class_weights': trial.suggest_categorical('auto_class_weights_to', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_ce = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_ce', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_ce', 2, 256),
            'loss_function': trial.suggest_categorical('loss_function_ce', ['RMSE', 'MAE']),
            'max_depth': trial.suggest_int('max_depth_ce', 2, 10),
            'iterations': trial.suggest_int('iterations_ce', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_ce', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_ce', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_ce', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_ce', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_ce', 1, 250),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_te = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_te', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_te', 2, 256),
            'loss_function': trial.suggest_categorical('loss_function_te', ['RMSE', 'MAE']),
            'max_depth': trial.suggest_int('max_depth_te', 2, 10),
            'iterations': trial.suggest_int('iterations_te', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_te', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_te', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_te', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_te', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_te', 1, 250),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        focal_alpha_to = trial.suggest_float('focal_alpha_to', 0.1, 0.9, step=0.1)
        focal_gamma_to = trial.suggest_float('focal_gamma_to', 0.5, 3, step=0.5)

        focal_alpha_co = trial.suggest_float('focal_alpha_co', 0.1, 0.9, step=0.1)
        focal_gamma_co = trial.suggest_float('focal_gamma_co', 0.5, 3, step=0.5)

        params_to['loss_function'] = f"Focal:focal_alpha={focal_alpha_to};focal_gamma={focal_gamma_to}"
        params_co['loss_function'] = f"Focal:focal_alpha={focal_alpha_co};focal_gamma={focal_gamma_co}"

        metrics = []

        for (train_ix, val_ix), propensity_model in zip(cv_idxs, propensity_models):
            x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
            x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

            if len(string_cols_to_encode):
                encoder = TargetEncoder()
                x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
                x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

            model = BaseXClassifier(
                control_outcome_learner=CatBoostClassifier(**params_co),
                treatment_outcome_learner=CatBoostClassifier(**params_to),
                control_effect_learner=CatBoostRegressor(**params_ce),
                treatment_effect_learner=CatBoostRegressor(**params_te)
            )

            p = propensity_model.predict(x_train[features])
            
            model.fit(X=x_train[features].values, treatment=x_train[config.dataset.treatment_col], y=y_train.ravel(), p=p)

            uplift_train = model.predict(x_train[features].values, p=propensity_model.predict(x_train[features]))
            uplift_val = model.predict(x_val[features].values, p=propensity_model.predict(x_val[features]))

            if all(uplift_val == 0):
                raise optuna.TrialPruned()

            metric_train = metric_func(y_train.values, uplift_train.ravel(), x_train[config.dataset.treatment_col].values)
            metric_val = metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values)

            metric = metric_stability(metric_train, metric_val, 0.2)
            metrics.append(metric)
            
        _logger.info(f'Trial #{trial.number}. Metrics: {metrics} (mean={np.mean(metrics):.4f}). Params: {trial.params}')
        return np.mean(metrics)

In [37]:
if config.general.optimize:
    _logger.info(f'Optimizing choosen model: {choosen_model}')

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(
            n_startup_trials=int(config.optimization.n_trials*(config.optimization.p_warmup/100)),
            multivariate=True
        )
    )
    try:
        study.optimize(objective, n_trials=config.optimization.n_trials, n_jobs=4)
    except KeyboardInterrupt:
        _logger.info('Оптимизация прервана пользователем')
    finally:
        save_params(study, optimize_path)
        clear_output()

In [38]:
with open(project_root / 'artifacts' / config.general.experiment_name / 'status.json', 'w') as f:
    status = {
        'init': True,
        'dataset': True,
        'features': True,
        'choose_model': True,
        'optimization': True,
        'fitting': False
    }
    json.dump(status, f)

In [ ]:
os._exit(00)